In [11]:
## here we are going to perform regression - where Estimated salary will be an output fetaure
## import all the necessary libraries 
import numpy as np 
import pandas as pd 
import tensorflow as ts 
import pickle 
import warnings 
warnings.filterwarnings("ignore") 
from sklearn.model_selection import train_test_split

In [4]:
data =pd.read_csv("Churn_Modelling.csv") 
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
## preprocess the data  
# drop irrelevent feartures 
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1) 
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
## sepearate the dependent and independent features 
X = data.drop("EstimatedSalary",axis=1) 
y = data["EstimatedSalary"]

In [7]:
X.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,619,France,Female,42,2,0.00,1,1,1,1
1,608,Spain,Female,41,1,83807.86,1,0,1,0
2,502,France,Female,42,8,159660.80,3,1,0,1
3,699,France,Female,39,1,0.00,2,0,0,0
4,850,Spain,Female,43,2,125510.82,1,1,1,0


In [8]:
y.head()

0    101348.88
1    112542.58
2    113931.57
3     93826.63
4     79084.10
Name: EstimatedSalary, dtype: float64

In [9]:
categorical_features = X.select_dtypes(include='object').columns 
numerical_features =X.select_dtypes(exclude='object').columns 
print("Categorical Features:",categorical_features, "\n")
print("Numerical Features:",numerical_features) 

Categorical Features: Index(['Geography', 'Gender'], dtype='str') 

Numerical Features: Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'Exited'],
      dtype='str')


In [12]:
# 1. Split the data FIRST because it prevent from the data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((8000, 10), (2000, 10), (8000,), (2000,))

In [16]:
## Now applying column Transformer  
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder 


Ordinal_encoder_gender = OrdinalEncoder() 
onehot_encoder_geo = OneHotEncoder(sparse_output=False) 
scaler = StandardScaler()

## creating a pipeline
preprocessor = ColumnTransformer(
    transformers=[

        ("OrdinalEncoder",Ordinal_encoder_gender,["Gender"]),
        ("OneHotEncoder",onehot_encoder_geo,["Geography"]), 
        
        #scaling is done only on continuos numerical values not categorical or encoded values like 0 and 1
        ("StandardScaler",scaler,['CreditScore',
            'Age',
            'Tenure',
            'Balance',
            'NumOfProducts',
            ])
    ], 
    remainder="passthrough", ## importatnt because it doesnot drop other remaing num features
    verbose_feature_names_out=False ## features k aage se remainder word hatane k liye dataframe me convert krne k baad
)

In [17]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [18]:
## convert into the dataframe
columns = preprocessor.get_feature_names_out()

X_train = pd.DataFrame(X_train, columns=columns)

X_test = pd.DataFrame(X_test, columns=columns)

In [19]:
print("Training Data:")
display(X_train.head())

print("Testing Data:")
display(X_test.head())

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Data:


,Gender,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,1.0,1.0,0.0,0.0,0.356500,-0.655786,0.345680,-1.218471,0.808436,1.0,1.0,0.0
1,1.0,0.0,1.0,0.0,-0.203898,0.294938,-0.348369,0.696838,0.808436,1.0,1.0,0.0
2,1.0,0.0,0.0,1.0,-0.961472,-1.416365,-0.695393,0.618629,-0.916688,1.0,0.0,1.0
3,0.0,1.0,0.0,0.0,-0.940717,-1.131148,1.386753,0.953212,-0.916688,1.0,0.0,1.0
4,1.0,1.0,0.0,0.0,-1.397337,1.625953,1.386753,1.057449,-0.916688,0.0,0.0,1.0


Testing Data:


,Gender,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,1.0,0.0,1.0,0.0,-0.577496,-0.655786,-0.695393,0.329937,0.808436,0.0,0.0,0.0
1,1.0,1.0,0.0,0.0,-0.297297,0.390011,-1.389442,-1.218471,0.808436,1.0,1.0,0.0
2,0.0,0.0,0.0,1.0,-0.525607,0.485083,-0.348369,-1.218471,0.808436,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,-1.511492,1.911170,1.039728,0.689272,0.808436,1.0,1.0,0.0
4,0.0,0.0,0.0,1.0,-0.951094,-1.131148,0.692704,0.782839,-0.916688,1.0,1.0,0.0


Training Shape: (8000, 12)
Testing Shape: (2000, 12)


In [20]:
## saving into pickle
import pickle 
with open('regression_preprocessor.pkl',"wb") as file: 
    pickle.dump(preprocessor,file)

## ANN model

In [22]:
## import Tensorflow libraries  
import tensorflow as tf 
from tensorflow.keras.models import Sequential  #ANN Is sequential means interconnected 
from tensorflow.keras.layers import Dense        # Dense means how much neurone will use in 1 Hidden layers 
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard 


In [ ]:
## Build our Ann model 
model = Sequential( [
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),   ##HL1 Connected with input Layer 
    Dense(32,activation='relu'),    #Hl2 created 
    Dense(1)   #output layer for regression by default it uses linear activation function
]
)

In [43]:
## before compiling the model best practice is you should define your own requirement  
opt = tf.keras.optimizers.Adam(learning_rate=0.1) 
loss =tf.keras.losses.mean_absolute_error

In [37]:
## compile model =  forward and backward propagation 
model.compile( 
    optimizer =opt,
    loss = loss, 
    metrics = ['mae']
)

In [38]:
## how many numbers of paramerter are there for training 
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [39]:
## set up early stopping  
from tensorflow.keras.callbacks import EarlyStopping 

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# EarlyStopping: Training ko unnecessary continue hone se rokta hai.

In [40]:
# create the path for monitoring the activivty  
import os 
import datetime 

log_dir = os.path.join("regressionlogs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")) 

## set up Tensorboard for training visualization and also configure callback  
from tensorflow.keras.callbacks import TensorBoard

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [45]:
## Finally the model Training  
history = model.fit( 
    X_train,
    y_train, 
    validation_data = (X_test,y_test),
    epochs =100, 
    callbacks = [tensorboard_callback,early_stopping]
)

Epoch 1/100
250/250 [==============================] - 0s 2ms/step - loss: 100430.2734 - mae: 100430.2734 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 100430.2734 - mae: 100430.2734 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 100430.2734 - mae: 100430.2734 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 100430.2578 - mae: 100430.2578 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 100430.2656 - mae: 100430.2656 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 100430.3125 - mae: 100430.3125 - val_loss: 98725.0625 - val_mae: 98725.0625
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 100430.2812 - mae: 

In [46]:
model.save('regression_model.h5')

In [47]:
model.evaluate(X_test, y_test)

63/63 [==============================] - 0s 799us/step - loss: 98725.0625 - mae: 98725.0625


[98725.0625, 98725.0625]

In [48]:
## LOad tensorboard Extension 
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [35]:
%tensorboard --logdir regressionlogs/

Reusing TensorBoard on port 6008 (pid 17552), started 0:00:16 ago. (Use '!kill 17552' to kill it.)